# Data Pipeline for Senate LDA Expenses
---

This is a data pipe designed to be used to pull out and sum all Senate LDA expense data while removing duplicates created by amendments or double-filling. The data at the end is very similar to Open secrets data, though the Open Secrets lobbying page shows totals that include the income statements from lobbying firms and I don't know much about those so I've left them out to be added at some future date. 

## Input Requirements
- To run this pipeline you will need to first get a csv from lobbyfinder.py
- This pipeline is designed to use all of the inbuild column names from lobbyfinder.py so I don't think it will work with anything else

## To Use
- Make sure all requirements are installed, as I installed the ones I needed as I went
- import your .csv into this folder and change the pdf.readcsv to your .csv name


## Output
- This will leave you with 4 graphics in the Viz folder all built with ploty so that you can interact with them in the notebook to check values and dates. 

### Libraries Used
- pandas, plotly, kaledio, nbformat

### DataFrames in this Pipeline

| DataFrame | Created From | Purpose |
|---|---|---|
| `df` | CSV file | Raw data — untouched source |
| `df_clean` | `df` | Filtered to `role == 'registrant'` only |
| `dupes` | `df_clean` | Diagnostic view — shows quarters with more than one filing |
| `df_deduped` | `df_clean` | Primary clean df — duplicates removed, keeping latest `dt_posted` per client/year/quarter |
| `df_yearly` | `df_deduped` | Expenses summed by company + year — feeds per-company line chart |
| `df_pct` | `df_yearly` | Adds year-over-year % change per company — feeds % change line chart |
| `df_total` | `df_deduped` | All companies

---
### Last Update: 04/26/26
---

## Step 1: dependances and importing the data

In [18]:
%pip install plotly

import sys
!{sys.executable} -m pip install kaleido

import plotly.io as pio
pio.kaleido.scope.mathjax = None



Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
C:\Users\craig\AppData\Local\Temp\ipykernel_26156\1501935392.py:7: DeprecationWarning: 
Use of plotly.io.kaleido.scope.mathjax is deprecated and support will be removed after September 2025.
Please use plotly.io.defaults.mathjax instead.

  pio.kaleido.scope.mathjax = None


In [19]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [20]:
df = pd.read_csv('lobbying_filings_Primes.csv')
df.head()

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,url
0,Senate LDA,Lockheed,client,49117f1a-a236-4583-a60e-919cc59e79c4,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,THE RHOADS GROUP,6644,LOCKHEED MARTIN CORP,108908,120000.0,NaN,Defense; Telecommunications,https://lda.senate.gov/filings/public/filing/4...
1,Senate LDA,Lockheed,client,31f28e72-7236-4837-b0b6-8d262d1d6cd8,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,VAN SCOYOC ASSOCIATES,39837,LOCKHEED MARTIN MISSILES & SPACE CO,147700,20000.0,NaN,Defense; Budget/Appropriations,https://lda.senate.gov/filings/public/filing/3...
2,Senate LDA,Lockheed,client,5d60eeb6-241e-4977-b0d1-852a36436cbf,1999,NaN,Mid-Year Termination (No Activity),1999-08-13T00:00:00-04:00,"CARTER GROUP, LLC",49008,LOCKHEED MARTIN AIRCRAFT & LOGISTICS CENTERS,156072,NaN,NaN,NaN,https://lda.senate.gov/filings/public/filing/5...
3,Senate LDA,Lockheed,client,d52d60e0-0536-4207-8d87-9c1084e31025,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,O'MELVENY & MYERS LLP,29904,LOCKHEED MARTIN CORP,135265,100000.0,NaN,Aerospace,https://lda.senate.gov/filings/public/filing/d...
4,Senate LDA,Lockheed,client,4e6ca9fd-7bad-451e-8a13-d7a698af4815,1999,NaN,Year-End Report,2000-02-14T00:00:00-05:00,DAP & ASSOC,48986,LOCKHEED MARTIN CORP,156067,120000.0,NaN,Telecommunications; Communications/Broadcastin...,https://lda.senate.gov/filings/public/filing/4...


## Step 2 - Data Cleaning

In [21]:
# Filter for [role] = registrant 

df_clean = df[df['role'] == 'registrant']


In [22]:
# Gather Duplicate Quarter reports & Amendments

dupes = df_clean[df_clean.duplicated(subset=['client_name', 'filing_year', 'filing_type'], keep=False)]
dupes[['client_name', 'filing_year', 'filing_type', 'dt_posted']].sort_values(['client_name', 'filing_year', 'filing_type'])


,client_name,filing_year,filing_type,dt_posted
4018,BOEING COMPANY,2000,Registration - Amendment,2000-08-11T00:00:00-04:00
4019,BOEING COMPANY,2000,Registration - Amendment,2000-11-08T00:00:00-05:00
4021,BOEING COMPANY,2000,Registration - Amendment,2001-02-14T00:00:00-05:00
4027,BOEING COMPANY,2003,Registration - Amendment,2003-02-13T00:00:00-05:00
4028,BOEING COMPANY,2003,Registration - Amendment,2003-05-21T00:00:00-04:00
4044,BOEING COMPANY,2007,Registration - Amendment,2007-03-22T11:53:25-04:00
4048,BOEING COMPANY,2007,Registration - Amendment,2008-06-08T00:00:00-04:00
3314,GENERAL DYNAMICS CORP,2005,Mid-Year Report,2005-08-12T00:00:00-04:00
3315,GENERAL DYNAMICS CORP,2005,Mid-Year Report,2006-02-14T00:00:00-05:00
3343,GENERAL DYNAMICS CORP,2013,1st Quarter - Report,2013-04-22T17:30:38.117000-04:00


In [23]:
# View Total year for identified dups

df_clean[(df_clean['client_name'] == 'AEROVIRONMENT INC.') & (df_clean['filing_year'] == 2009)]

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,url


In [24]:
# Creating a new column for base_quarter, and removing dups based on dt_posted datetime

df_clean['base_quarter'] = df_clean['filing_type'].str.replace(r'\s*-\s*(Report|Amendment).*', '', regex=True).str.strip()

df_deduped = (
    df_clean.sort_values('dt_posted', ascending=False)
            .drop_duplicates(subset=['client_name', 'filing_year', 'base_quarter'], keep='first')
            .reset_index(drop=True)
)


In [25]:
df_deduped[(df_deduped['client_name'] == 'AEROVIRONMENT INC.') & (df_deduped['filing_year'] == 2010)]

,source,query_company,role,filing_uuid,filing_year,period,filing_type,dt_posted,registrant_name,registrant_id,client_name,client_id,income,expenses,lobbying_issues,url,base_quarter


In [26]:
# Sum the expenses for the deduped years

df_deduped.groupby('filing_year')['expenses'].sum().reset_index().style.format({'expenses': '${:,.0f}'})



,filing_year,expenses
0,1999,"$29,617,104"
1,2000,"$36,553,915"
2,2001,"$38,698,941"
3,2002,"$45,505,116"
4,2003,"$49,017,896"
5,2004,"$40,145,611"
6,2005,"$44,709,545"
7,2006,"$62,148,340"
8,2007,"$53,233,298"
9,2008,"$76,494,813"


In [27]:
df_deduped.groupby(['client_name', 'filing_year'])['expenses'].sum().reset_index().style.format({'expenses': '${:,.0f}'})


,client_name,filing_year,expenses
0,BAE SYSTEMS INC,1999,"$200,000"
1,BAE SYSTEMS INC,2000,"$480,000"
2,BAE SYSTEMS INC,2001,"$920,000"
3,BAE SYSTEMS INC,2002,"$1,600,000"
4,BAE SYSTEMS INC,2003,"$1,620,000"
5,BAE SYSTEMS INC,2004,"$1,420,000"
6,BAE SYSTEMS INC,2005,"$4,000,000"
7,BAE SYSTEMS INC,2006,"$8,420,000"
8,BAE SYSTEMS INC,2007,"$4,160,000"
9,BAE SYSTEMS INC,2008,"$5,230,000"


## Step 3 - Vizualizations 

In [28]:
# installing additional dependancies 

import sys
!{sys.executable} -m pip install --upgrade nbformat

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
# Graphing each companies summed expenses per year

df_yearly = df_deduped.groupby(['client_name', 'filing_year'])['expenses'].sum().reset_index()

fig = px.line(df_yearly, x='filing_year', y='expenses', color='client_name', markers=True,
              labels={'filing_year': 'Year', 'expenses': 'Lobbying Expenses', 'client_name': 'Company'})
fig.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig.update_layout(title='Total Company Lobbying Expenses by Year')

fig.show()

fig.write_image('Viz/Company_change_expenses_by_year.svg', scale=2)



In [30]:
# creating the % change dataset

df_pct = (
    df_deduped.groupby(['client_name', 'filing_year'])['expenses'].sum()
    .reset_index()
    .sort_values(['client_name', 'filing_year'])
)

df_pct['pct_change'] = df_pct.groupby('client_name')['expenses'].pct_change() * 100
df_pct['pct_change'] = df_pct['pct_change'].fillna(0)


df_pct.head(20)


,client_name,filing_year,expenses,pct_change
0,BAE SYSTEMS INC,1999,200000.0,0.000000
1,BAE SYSTEMS INC,2000,480000.0,140.000000
2,BAE SYSTEMS INC,2001,920000.0,91.666667
3,BAE SYSTEMS INC,2002,1600000.0,73.913043
4,BAE SYSTEMS INC,2003,1620000.0,1.250000
5,BAE SYSTEMS INC,2004,1420000.0,-12.345679
6,BAE SYSTEMS INC,2005,4000000.0,181.690141
7,BAE SYSTEMS INC,2006,8420000.0,110.500000
8,BAE SYSTEMS INC,2007,4160000.0,-50.593824
9,BAE SYSTEMS INC,2008,5230000.0,25.721154


In [31]:
# Graphing %change by company

fig = px.line(df_pct, x='filing_year', y='pct_change', color='client_name', markers=True,
              labels={'filing_year': 'Year', 'pct_change': '% Change in Expenses', 'client_name': 'Company'})
fig.add_hline(y=0, line_dash='dash', line_color='grey')
fig.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig.update_layout(title='% Change in Lobbying Expenses by Year')

fig.write_image('Viz/%_change_expenses_by_year.svg', scale=2)

fig.show()


In [32]:
# showing totals

df_total = df_deduped.groupby('filing_year')['expenses'].sum().reset_index()
df_total['pct_change'] = df_total['expenses'].pct_change() * 100
df_total['pct_change'] = df_total['pct_change'].fillna(0)

fig1 = px.line(df_total, x='filing_year', y='expenses', markers=True,
               title='Total Lobbying Expenses by Year',
               labels={'filing_year': 'Year', 'expenses': 'Total Expenses'})
fig1.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))
fig1.show()

fig2 = px.line(df_total, x='filing_year', y='pct_change', markers=True,
               title='% Change in Total Lobbying Expenses by Year',
               labels={'filing_year': 'Year', 'pct_change': '% Change'})
fig2.add_hline(y=0, line_dash='dash', line_color='grey')
fig2.update_layout(hoverlabel=dict(bgcolor='white', font_color='black', bordercolor='lightgrey'))


fig1.write_image('Viz/total_expenses_by_year.svg', scale=2)
fig2.write_image('Viz/pct_change_total_expenses.svg', scale=2)

fig2.show()